# FRV001 · 日内与日频完整研究

这个 Notebook 运行固定的 5 分钟与 5 交易日反转规则。两个版本都按品种独立交易一手，并检查退出来自信号过零、由 `n` 推导的五根期限，还是主连换月。

In [ ]:
import pandas as pd
from infra.config import PROJECT, factor_strategies, product_ids
from infra.runner import DEFAULT_END, DEFAULT_START, DEFAULT_WARMUP, run_factor

FACTOR_ID = 'FRV001'
RUN_DIR = PROJECT / FACTOR_ID.lower() / 'runs' / 'v3_10y'
PRODUCTS = product_ids()

## 1. 固定参数

In [ ]:
pd.DataFrame(factor_strategies(FACTOR_ID)).T

## 2. 完整十年运行

In [ ]:
summary = run_factor(FACTOR_ID, PRODUCTS, warmup_start=DEFAULT_WARMUP, start=DEFAULT_START, end_exclusive=DEFAULT_END)
summary

## 3. 单品种 PnL 与完整交易表

In [ ]:
strategy_id = 'FRV001_daily_5d_v1'
product_id = 'SHFE.RB'
pnl = pd.read_csv(RUN_DIR / strategy_id / product_id / 'pnl.csv', parse_dates=['date'])
pnl.set_index('date')['cumulative_net_pnl'].plot(figsize=(13, 4), title=f'{strategy_id} · {product_id}')
trades = pd.read_csv(RUN_DIR / strategy_id / product_id / 'trades.csv')
display(trades.head())

## 4. 退出来自自然回补还是五根期限

In [ ]:
exit_report = trades.groupby('exit_reason').agg(trades=('trade_id', 'count'), net_pnl=('net_pnl', 'sum'))
exit_report['share'] = exit_report['trades'] / exit_report['trades'].sum()
exit_report